In [ ]:
! pip3 install -r requirements.txt

In [2]:
!nvidia-smi

Tue Nov 14 15:07:56 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.105.17   Driver Version: 525.105.17   CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            Off  | 00000000:00:04.0 Off |                    0 |
| N/A   67C    P0    29W /  70W |      2MiB / 15360MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  Tesla T4            Off  | 00000000:00:05.0 Off |                    0 |
| N/A   

In [ ]:
import os
from utils.utils import get_offers
from utils.secrets_utils import get_secret
from utils.constants import GCP_PROJECT_ID,ENV_SHORT_NAME,BIGQUERY_ANALYTICS_DATASET
import vertexai
from google.cloud import aiplatform
import json
from vertexai.language_models import ChatModel, InputOutputTextPair


passculture-data-ehp.analytics_dev.enriched_item_metadata


In [ ]:
offer_list = get_offers(number_offers=500,analytics_dataset="analytics_stg")

/home/airflow/.local/lib/python3.10/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Downloading: 100%|██████████|


In [ ]:
# Cloud project id.
PROJECT_ID = GCP_PROJECT_ID,ENV_SHORT_NAME # @param {type:"string"}

# The region you want to launch jobs in.
# Select region based on the accelerators and regions supported by Vertex AI Prediction
# https://cloud.google.com/vertex-ai/docs/predictions/configure-compute.
REGION = "us-central1"  # @param {type:"string"}

# The Cloud Storage bucket for storing experiments output.
# Start with gs:// prefix, e.g. gs://foo_bucket.
BUCKET_URI = "gs://data-bucket-stg"  # @param {type:"string"}

! gcloud config set project $GCP_PROJECT_ID


STAGING_BUCKET = os.path.join(BUCKET_URI, "temporal")

# The service account looks like:
# '@.iam.gserviceaccount.com'
# Please go to https://cloud.google.com/iam/docs/service-accounts-create#iam-service-accounts-create-console
# and create service account with `Vertex AI User` and `Storage Object Admin` roles.
# The service account for deploying fine tuned model.
SERVICE_ACCOUNT = "cloud-run-dev@passculture-data-ehp.iam.gserviceaccount.com"  # @param {type:"string"}


To update your Application Default Credentials quota project, use the `gcloud auth application-default set-quota-project` command.
Updated property [core/project].


In [ ]:
aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=STAGING_BUCKET)

In [ ]:
# %%time
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda"  # the device to load the model onto
model_name = "mistralai/Mistral-7B-v0.1" # "mistralai/Mistral-7B-Instruct-v0.1" -> specialized in english and code
model = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="auto", return_dict=True, torch_dtype=torch.float16
)





Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

CPU times: user 44.2 s, sys: 35.7 s, total: 1min 19s
Wall time: 1min 18s


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
pipeline = transformers.pipeline("text-generation", model=model, tokenizer=tokenizer)


In [ ]:


prompt = "My favourite condiment is"

sequences = pipeline(
    prompt,
    max_length=200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for seq in sequences:
    print(f"Result: {seq['generated_text']}")

### https://huggingface.co/blog/accelerate-large-models